In [1]:
import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np
import pandas as pd
from tqdm import tqdm
from datetime import datetime
import os
import array
import collections

from typing import Dict, List, Optional, Text, Tuple

In [2]:
max_port_size = 50

In [3]:
retriever_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_07_01_31\retriever_v3_port_v2__fixed_max_port_size_50"
ranking_location_ = r"D:\dev work\recommender systems\Atrad_CARS\model_weights\2024_05_27\tf_listwise_ranking_2024_05_27_11_20"
stock_info_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\stock_data.xlsx"

train_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\retriver_train".format(max_port_size)
test_ds_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\retriver_test".format(max_port_size)
portfolios_loc = r"D:\dev work\recommender systems\Atrad_CARS\data\portfolios_v2_fixed_max_port_size_{}_useridseq\portfolios".format(max_port_size)

results_loc = r"D:\dev work\recommender systems\Atrad_CARS\results"

In [4]:
test_ds = tf.data.Dataset.load(test_ds_loc).cache()

train_ds = tf.data.Dataset.load(train_ds_loc).cache()

portfolios = tf.data.Dataset.load(portfolios_loc).cache()

In [5]:
len(train_ds)

85393

In [6]:
from retrieval_recommender_v3 import Retriever

retriever = Retriever(
    portfolios = portfolios
)

retriever.load_weights(retriever_location_)

retriever.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))


In [7]:
from ranker_recommender import Ranker

ranker = Ranker(
    loss = tf.keras.losses.MeanSquaredError(),
    portfolios = portfolios
)

ranker.load_weights(ranking_location_)
ranker.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=0.1))


In [8]:
stock_info = pd.read_excel(stock_info_loc)
stock_info = stock_info.drop(['Unnamed: 0','buisnesssummary'],axis = 1)
stock_info = stock_info.rename(columns = {
    'symbol':'STOCKCODE',
    'name' : 'STOCKNAME',
    'gics_code' : 'GICS'
})
stock_info = stock_info[~stock_info['GICS'].isna()]

stock_info.shape
print("items data shape :: {}".format(stock_info.shape))
unique_items_ = np.unique(np.concatenate(list(train_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator())))
stock_info = stock_info[stock_info['STOCKCODE'].isin([item.decode('utf-8') for item in unique_items_])]

items_ds = tf.data.Dataset.from_tensor_slices(stock_info.to_dict(orient= 'list'))

items data shape :: (280, 3)


In [9]:
train_user_to_items = collections.defaultdict(lambda: array.array("i"))
train_user_to_items['seq1'].append(99)
train_user_to_items['seq1'].append(11)

train_user_to_items['seq2'].append(99)
train_user_to_items['seq2'].append(11)

In [10]:
train_user_to_items

defaultdict(<function __main__.<lambda>()>,
            {'seq1': array('i', [99, 11]), 'seq2': array('i', [99, 11])})

In [11]:
user_id, items = train_user_to_items.items()
user_id, items

(('seq1', array('i', [99, 11])), ('seq2', array('i', [99, 11])))

In [12]:
for user_id, test_items in tqdm(train_user_to_items.items()):
    print(user_id)

100%|██████████| 2/2 [00:00<?, ?it/s]

seq1
seq2


In [13]:
item_ids = np.concatenate(list(items_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator()))

item_vocabulary = dict(zip(item_ids.tolist(), range(len(item_ids))))
item_vocabulary_inv = {v: k for k, v in item_vocabulary.items()}

train_user_to_items = collections.defaultdict(lambda: array.array("i"))
test_user_to_items = collections.defaultdict(lambda: array.array("i"))

cdsaccno_to_user_id = dict() #collections.defaultdict(lambda: array.array("i"))

for row in train_ds.as_numpy_iterator():
    user_id = row["USER_ID"]
    cdsaccno = row["CDSACCNO"]
    item_id = item_vocabulary[row["STOCKCODE"]]
    train_user_to_items[cdsaccno].append(item_id)
    cdsaccno_to_user_id[cdsaccno] = user_id

In [14]:
for row in test_ds.as_numpy_iterator():
    # user_id = row["USER_ID"]
    cdsaccno = row["CDSACCNO"]
    item_id = item_vocabulary[row["STOCKCODE"]]
    test_user_to_items[cdsaccno].append(item_id)

In [15]:
cdsaccno_to_user_id

{b'BMS-10544-LC/00': array([b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'RIL', b'SLTL', b'RCL',
        b'CALT', b'PLC'], dtype=object),
 b'BMS-11214-LC/00': array([b'LIOC', b'HELA', b'AAIC', b'MELS', b'NTB', b'MASK', b'SLTL',
        b'MGT', b'CFVF', b'CIC'], dtype=object),
 b'BMS-11807-LC/00': array([b'BIL', b'AEL', b'RCL', b'HAYL', b'VONE', b'TJL', b'ACL', b'TKYO',
        b'MGT', b'SEYB'], dtype=object),
 b'BMS-11829-LI/00': array([b'SCAP', b'UBC', b'RICH', b'LWL', b'CFVF', b'COMB', b'TILE',
        b'CCS', b'LPL', b'RCL'], dtype=object),
 b'BMS-12282-LI/00': array([b'EXPO', b'DIAL', b'VONE', b'TKYO', b'PLR', b'EDEN', b'COOP',
        b'TESS', b'BOGA', b'EML'], dtype=object),
 b'BMS-12407-LI/00': array([b'SUN', b'LWL', b'PACK', b'CTBL', b'LALU', b'LIOC', b'ASIY',
        b'CIC', b'SLTL', b'AGST'], dtype=object),
 b'BMS-12476-LI/00': array([b'JAT', b'HAYC', b'HAYL', b'HELA', b'DIST', b'DOCK', b'CARG',
        b'LOLC', b'SAMP', b'BIL'], dtype=object),
 b'BMS-12791-LI/00': array([b'LO

In [16]:
# test_user_to_items

In [17]:
{'USER_ID' : tf.constant([cdsaccno_to_user_id[b'BMS-10544-LC/00']])}

{'USER_ID': <tf.Tensor: shape=(1, 10), dtype=string, numpy=
 array([[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'RIL', b'SLTL',
         b'RCL', b'CALT', b'PLC']], dtype=object)>}

In [18]:
user_embedding = retriever.user_model(
      {
        'USER_ID' : tf.constant([cdsaccno_to_user_id[b'BMS-10544-LC/00']])
      }
      ).numpy()

In [19]:
user_embedding

array([[-0.11184297,  0.09500204, -0.10830786,  0.18936871,  0.05952095,
        -0.09766227, -0.27902097,  0.0280554 ,  0.40339804,  0.14873774,
         0.23176849,  0.09467231,  0.09489461, -0.02286706, -0.16975546,
        -0.16981795, -0.22439611,  0.45983496,  0.05549452, -0.07910415,
         0.67153895, -0.25654933,  0.34431496, -0.27195838,  0.01073585,
         0.04809446,  0.16255832, -0.0346479 ,  0.24187016, -0.1500047 ,
        -0.21826907, -0.01783906]], dtype=float32)

# evaluation function

In [20]:
def evaluate(retriever,
             test: tf.data.Dataset,
             train: Optional[tf.data.Dataset] = None,
             timestamp: int = datetime.timestamp(datetime.now()),
             k: int = 10):
  
  item_ids = np.concatenate(list(items_ds.batch(1000).map(lambda x: x["STOCKCODE"]).as_numpy_iterator()))

  item_vocabulary = dict(zip(item_ids.tolist(), range(len(item_ids))))
  item_vocabulary_inv = {v: k for k, v in item_vocabulary.items()}

  train_user_to_items = collections.defaultdict(lambda: array.array("i"))
  test_user_to_items = collections.defaultdict(lambda: array.array("i"))

  cdsaccno_to_user_id = dict() #collections.defaultdict(lambda: array.array("i"))

  if train is not None:
    for row in train.as_numpy_iterator():
      user_id = row["USER_ID"]
      cdsaccno = row["CDSACCNO"]
      item_id = item_vocabulary[row["STOCKCODE"]]
      train_user_to_items[cdsaccno].append(item_id)
      cdsaccno_to_user_id[cdsaccno] = user_id

  for row in test.as_numpy_iterator():
    # user_id = row["USER_ID"]
    cdsaccno = row["CDSACCNO"]
    item_id = item_vocabulary[row["STOCKCODE"]]
    test_user_to_items[cdsaccno].append(item_id)

  item_embeddings = np.concatenate(list(items_ds.batch(len(items_ds)).map(lambda x: retriever.item_model(x)).as_numpy_iterator()))

  user_ids = []
  cdsaccnos = []
  precision_values = []
  recall_values = []
  num_test_items = []
  num_train_items = []
  recommendations = []

  for cdsaccno, test_items in tqdm(test_user_to_items.items()):
    user_embedding = retriever.user_model(
      {
        'USER_ID' : tf.constant([cdsaccno_to_user_id[cdsaccno]])
      }
      ).numpy()
    scores = (user_embedding @ item_embeddings.T).flatten()

    test_items = np.frombuffer(test_items, dtype=np.int32)
    
    if train is not None:
      train_items = np.frombuffer(
          train_user_to_items[cdsaccno], dtype=np.int32)
      scores[train_items] = -1e6

    
    # print(cdsaccno)
    
    top_items = np.argsort(-scores)[:k]
    recommendations.append([item_vocabulary_inv[item_id].decode('utf-8') for item_id in top_items])

    num_test_items_in_k = sum(x in top_items for x in  test_items)
    precision_values.append(num_test_items_in_k / k)
    
    recall_values.append(num_test_items_in_k / len(test_items))
    num_test_items.append(len((test_items)))
    num_train_items.append(len(train_user_to_items[cdsaccno]))
    cdsaccnos.append(cdsaccno)
    user_ids.append(cdsaccno_to_user_id[cdsaccno])


  # print(cdsaccno)
  results_df_ = pd.DataFrame(
    columns = ['CDSACCNO','USER_ID','precision@k', 'recall@k','num_test_items','portfolio_size', 'recommendations'],
    data = list(zip(cdsaccnos, user_ids, precision_values, recall_values, num_test_items, num_train_items, recommendations))
  )

  # print(results_df_.head())

  return {
      "precision_at_k": np.mean(precision_values),
      "recall_at_k": np.mean(recall_values),
      "results_df_" : results_df_
  }

In [21]:
results = evaluate(
    retriever,
    test_ds,
    train_ds
)

100%|██████████| 2984/2984 [00:04<00:00, 693.02it/s]


In [22]:
results['precision_at_k'] , results['recall_at_k']

(0.04638069705093834, 0.09277815013404826)

In [23]:
results_df_ = results['results_df_']
results_df_['CDSACCNO'] = results_df_['CDSACCNO'].apply(lambda x: x.decode('utf-8'))
results_df_

,CDSACCNO,USER_ID,precision@k,recall@k,num_test_items,portfolio_size,recommendations
0,BMS-10544-LC/00,"[b'COCO', b'EMER', b'SAMP', b'NDB', b'EML', b'...",0.0,0.0,5,23,"[RAL, CWM, DPL, SLND, IDL, ASCO, SHOT, ECL, MA..."
1,BMS-11214-LC/00,"[b'LIOC', b'HELA', b'AAIC', b'MELS', b'NTB', b...",0.0,0.0,5,45,"[CARE, KAHA, DPL, LHCL, LDEV, KGAL, RHTL, LLUB..."
2,BMS-11807-LC/00,"[b'BIL', b'AEL', b'RCL', b'HAYL', b'VONE', b'T...",0.1,0.2,5,17,"[CFIN, TYRE, ABAN, AHPL, CONN, TILE, HHL, HEXP..."
3,BMS-11829-LI/00,"[b'SCAP', b'UBC', b'RICH', b'LWL', b'CFVF', b'...",0.1,0.2,5,21,"[EBCR, UAL, CITH, BLUE, SHL, SAMP, CERA, SOY, ..."
4,BMS-12282-LI/00,"[b'EXPO', b'DIAL', b'VONE', b'TKYO', b'PLR', b...",0.0,0.0,5,24,"[MSL, ALHP, LCBF, MHDL, MFL, CSF, ASCO, BLI, B..."
...,...,...,...,...,...,...,...
2979,RPS-953190630-VN/00,"[b'BRWN', b'EXT', b'GREG', b'BFL', b'HVA', b'J...",0.0,0.0,5,45,"[SOY, RGEM, CIT, CPRT, OFEQ, LPRT, SFCL, RWSL,..."
2980,SBK-80957-LC/00,"[b'STAF', b'TAP', b'LWL', b'MGT', b'SAMP', b'S...",0.1,0.2,5,45,"[CFLB, HEXP, CONN, RCL, TILE, PABC, ABAN, JETS..."
2981,SCB-11576-LC/00,"[b'AEL', b'JAT', b'CALT', b'ACL', b'SUN', b'AL...",0.3,0.6,5,20,"[DIST, TYRE, LALU, MELS, DIAL, SIRA, LLUB, GRA..."
2982,SCB-11577-LC/00,"[b'HNB', b'AEL', b'JAT', b'CALT', b'SUN', b'AL...",0.2,0.4,5,18,"[DIAL, CTC, PABC, MELS, CFIN, LMF, NEST, HAYL,..."


In [24]:
results_df_['recall@k'].value_counts()

recall@k
0.00    1894
0.20     834
0.40     221
0.60      30
0.80       3
1.00       1
0.25       1
Name: count, dtype: int64

In [25]:
(results_df_[results_df_['recall@k'] != 0].shape[0] *100)/results_df_.shape[0]

36.52815013404826

In [33]:
results_loc_ = r"D:\dev work\recommender systems\Atrad_CARS\results"

retriever_name = os.path.basename(retriever_location_)
ranker_name = os.path.basename(ranking_location_)

results_file_name = retriever_name + "_&_" + ranker_name + "_results_fixed_port_{}.csv".format(max_port_size)

results_save_path = os.path.join(results_loc_, results_file_name)
print("saving results as @ {}".format(results_save_path))
results_df_.to_csv(results_save_path, index = False)

saving results as @ D:\dev work\recommender systems\Atrad_CARS\results\retriever_v3_port_v2__fixed_max_port_size_50_&_tf_listwise_ranking_2024_05_27_11_20_results_fixed_port_50.csv


In [27]:
len(train_ds), len(test_ds)

(85393, 14917)

In [28]:
unique_train_items = np.unique(np.concatenate(list(train_ds.batch(100).map(lambda x: x['STOCKCODE']))))

In [29]:
unique_test_items = np.unique(np.concatenate(list(test_ds.batch(100).map(lambda x: x['STOCKCODE']))))

In [30]:
set(unique_train_items) == set(unique_test_items), set(unique_train_items) - set(unique_test_items)

(False, {b'BLI', b'NEH'})

In [31]:
unique_train_items.shape, unique_test_items.shape

((268,), (266,))

In [32]:
results_df_['']

KeyError: ''